# Why does parity fail at frame 7 on a T4?

`gpu_parity.py` reports `range_image` differs at frame 7 on a Tesla T4, while
CPU and CUDA agree bit for bit on the laptop RTX 5050.

The key is `(pix << 49) | (range_bits << 18) | point_index`. The index is unique,
so every key is unique, the sort is a total order, and the winner per pixel is
deterministic **by construction** — ties cannot be the cause. The bit budget is
guarded at construction, so it is not an overflow either.

That leaves the float arithmetic that *produces* the key. This asks which:

* **(a) the winner moved** — the same points land in the pixel, but `r` differs by
  an ULP and reorders them; or
* **(b) a point moved pixels** — `az`/`el` differ by an ULP right at a bin edge,
  so `floor()` lands in a different bin.

They have different fixes, so the distinction is the whole point.

In [ ]:
# 1. The machine. This is the whole point of the notebook -- a DIFFERENT card.
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
import torch

print("torch", torch.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU -- set Accelerator to GPU in the sidebar"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {vram:.1f} GB")
assert vram > 12, f"{name} has {vram:.1f} GB -- the 16 GB contention test needs more than the laptop's 8"

In [ ]:
# 2. The repo. Public, so no token.
%cd /kaggle/working
!rm -rf vrgrid-26
!git clone -q https://github.com/Stxtics03/vrgrid-26.git
%cd /kaggle/working/vrgrid-26
!git log --oneline -1
!pip -q install -e . 2>&1 | tail -2
try:
    import cupy

    print("cupy", cupy.__version__, "(preinstalled)")
except ImportError:
    !pip -q install cupy-cuda12x

In [ ]:
# 4. Assemble the asset tree vrgrid expects. The mirror supplies sequences/; the
#    bundle supplies poses/ and the checkpoint.
#    loader.py uses the OFFICIAL KITTI GT poses at poses/<seq>.txt, NOT the
#    SemanticKITTI SLAM poses inside sequences/<seq>/poses.txt. Do not substitute
#    one for the other -- they are different quantities and the swap is silent.
import os
from pathlib import Path

# Print what actually mounted before asserting anything. A dataset that is
# attached but empty looks identical to one that is missing, three cells later.
IN = Path("/kaggle/input")
for root, dirs, files in os.walk(IN):
    depth = root.replace(str(IN), "").count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    print("  " * depth + os.path.basename(root) + "/")

def find(child, maxdepth=4):
    """The mounted dataset directory containing <child>.

    Kaggle nests these as /kaggle/input/datasets/<owner>/<slug>/, so a
    top-level scan finds only "datasets". Breadth-first with a depth cap --
    never rglob, the mirror alone is 96 GB and 66k files.
    """
    frontier = [IN]
    for _ in range(maxdepth):
        nxt = []
        for d in frontier:
            if (d / child).is_dir():
                return d
            try:
                nxt += [k for k in d.iterdir() if k.is_dir()]
            except (PermissionError, OSError):
                pass
        frontier = nxt
    return None

MIRROR = find("sequences")
BUNDLE = find("poses")
print("\nMIRROR:", MIRROR, "\nBUNDLE:", BUNDLE)
assert MIRROR, "no mounted dataset contains sequences/"
assert BUNDLE, "no mounted dataset contains poses/ -- check the Input panel"

A = Path("/kaggle/working/assets")
(A / "dataset").mkdir(parents=True, exist_ok=True)
for src, dst in [(MIRROR / "sequences", A / "dataset/sequences"),
                 (BUNDLE / "poses",     A / "dataset/poses"),
                 (BUNDLE / "checkpoints", A / "checkpoints")]:
    assert src.exists(), f"missing: {src}"
    if not dst.exists():
        dst.symlink_to(src)

os.environ["VRGRID_ASSETS"] = str(A)
os.environ["VRGRID_DATA_ROOT"] = str(A / "dataset")
os.environ["VRGRID_FRNET_CHECKPOINT"] = str(A / "checkpoints/frnet-semantickitti_seg.pth")
for k in ("VRGRID_ASSETS", "VRGRID_DATA_ROOT", "VRGRID_FRNET_CHECKPOINT"):
    print(f"{k}={os.environ[k]}")


In [ ]:
import pathlib

pathlib.Path('/kaggle/working/vrgrid-26/elprobe.py').write_text(r'''"""Which side moved? Elevation for frame 7's offending points, three ways.

Run on the laptop and on the Kaggle Xeon and diff. Nothing here touches the
GPU -- it isolates numpy's own float32 transcendentals, which are SIMD
dispatched and need not agree across CPU architectures.
"""
import numpy as np, platform
from vrgrid.perception import loader

IDX = [56230, 58375, 58376, 89740, 89741]
PHI_MAX, D_PHI = None, None
from vrgrid.perception import range_image as RI
cfg = RI.load_sensor_config()
d_theta, d_phi = RI.bin_widths(cfg)
phi_max = np.deg2rad(cfg["phi_max_deg"])

pts = None
for i, (p, l, pose) in enumerate(loader.scans("08", max_frames=8)):
    if i == 7:
        pts = p
        break

print(f"host: {platform.processor() or platform.machine()} | numpy {np.__version__}")
print(f"simd: {getattr(np.core, '_multiarray_umath', None) and 'n/a'}")
print(f"{'idx':>7} {'el_f32(numpy)':>22} {'el_f64->f32':>22} {'v_f32':>6} {'v_dbl':>6}")
for k in IDX:
    xyz = pts[k, :3]
    r32 = np.linalg.norm(xyz)                      # float32, as project() does
    zr32 = np.float32(xyz[2]) / r32
    el32 = np.arcsin(np.clip(zr32, -1.0, 1.0))     # numpy float32 arcsin

    x, y, z = map(np.float64, xyz)
    r64 = np.sqrt(x*x + y*y + z*z)
    el_narrowed = np.float32(np.arcsin(np.clip(z / r64, -1.0, 1.0)))  # the kernel's way

    v_f32 = int(np.floor((phi_max - el32) / d_phi))
    v_dbl = int(np.floor((phi_max - np.float64(el_narrowed)) / d_phi))
    print(f"{k:>7} {el32.item():>22.17g} {el_narrowed.item():>22.17g} {v_f32:>6} {v_dbl:>6}"
          + ("   <-- el DIFFERS" if el32 != el_narrowed else ""))
''')
print('wrote elprobe.py')

In [ ]:
# Pure numpy -- no GPU. Isolates whether numpy's own float32 arcsin
# bins these points the same way on a Xeon as it does on the laptop.
!cd /kaggle/working/vrgrid-26 && python elprobe.py 2>&1 | tee /kaggle/working/elprobe.log

In [ ]:
!lscpu | grep -E 'Model name|Flags' | head -2 | cut -c1-200